In [1]:
import pandas as pd
import hashlib
import json

def generate_hash(val):
    """
    Deterministic hash for strings and simple types.
    Handles None and non-string types safely.
    """
    if val is None:
        return None
    return hashlib.sha256(str(val).encode('utf-8')).hexdigest()

# 1. Configuration
dataset_base = "hf://datasets/yujunzhou/LabSafety_Bench/"
splits = {
    'QA': 'data/QA-00000-of-00001.parquet',
    'QA_I': 'data/QA_I-00000-of-00001.parquet'
}

# 2. Load the QA_I split (this specifically contains the image-based questions)
print("Loading dataset...")
df = pd.read_parquet(dataset_base + splits["QA_I"])

# 3. Filter for rows where 'Image Path' is not null/empty
# In this dataset, images are often stored in 'Image Path' or 'image' columns
df_with_images = df[df['Image Path'].notna()].copy()

# 4. Generate a unique ID/Hash for each image path for tracking
df_with_images['image_hash'] = df_with_images['Image Path'].apply(generate_hash)



Loading dataset...


In [2]:
df_with_images.columns

Index(['Question', 'Explanation', 'Correct Answer', 'Category', 'Topic',
       'Image Path', 'Level', 'Decoded Image', 'image_hash'],
      dtype='object')

In [3]:
import pandas as pd
import hashlib
import os
import re
import numpy as np

# --- Helper Functions ---

def generate_hash(val):
    if val is None:
        return None
    return hashlib.sha256(str(val).encode('utf-8')).hexdigest()

def extract_first_category(val):
    """
    Extracts the first category from a list or list-like string.
    """
    if val is None: return "Unknown"
    
    # If it is already a list object
    if isinstance(val, (list, pd.Series, np.ndarray)):
        return val[0] if len(val) > 0 else "Unknown"
    
    # If it is a string representation
    str_val = str(val).strip()
    if str_val.startswith('[') and str_val.endswith(']'):
        content = str_val[1:-1]
        if not content: return "Unknown"
        parts = content.split(',')
        return parts[0].strip().strip("'").strip('"')
            
    return str_val

import re

def parse_question_and_choices(text):
    """
    Splits a raw question string into the question body and a list of choices.
    Handles formats like:
      - "Question? A: Opt1 B: Opt2"
      - "Question?\nA: Opt1\nB: Opt2" (literal \n chars)
      - "Question? (A) Opt1 (B) Opt2"
    """
    if not isinstance(text, str):
        return str(text), []

    # 1. Detect the start of the choices (A)
    # Pattern explanation:
    # (?:\\n|\n|\s|^)  -> Non-capturing group matching: literal \n, actual newline, whitespace, or start of line
    # (A[:\.\)])       -> Capturing group 1: Matches 'A' followed by ':', '.', or ')'
    start_pattern = r'(?:\\n|\n|\s|^)(A[:\.\)])'
    
    match = re.search(start_pattern, text)
    
    if not match:
        return text, []

    # We use match.start(1) to get the index of "A", but we want to cut the question 
    # before the whitespace/newline preceding "A".
    # match.start() gives the index of the prefix (space/\n), which is usually where the question ends.
    split_index = match.start()
    
    question_part = text[:split_index].strip()
    choices_part = text[split_index:]

    # 2. Extract options A, B, C, D (and E if present)
    # Pattern explanation:
    # (?:^|\\n|\n|\s)      -> Prefix: start of string, newlines, or space
    # ([A-E])              -> Group 1: The letter (A-E)
    # [:\.\)]              -> The separator (:, ., or ))
    # \s* -> Optional whitespace
    # (.*?)                -> Group 2: The content (non-greedy)
    # (?=(?:\\n|\n|\s)[A-E][:\.\)]|$) -> Lookahead: Stop when we see the next Letter+Separator OR end of string
    choice_pattern = r'(?:^|\\n|\n|\s)([A-E])[:\.\)]\s*(.*?)(?=(?:\\n|\n|\s)[A-E][:\.\)]|$)'
    
    raw_choices = re.findall(choice_pattern, choices_part, re.DOTALL)
    
    # raw_choices will be a list of tuples: [('A', 'text'), ('B', 'text')...]
    # We only want the text content.
    clean_choices = [content.strip() for letter, content in raw_choices]
    
    # If the regex failed to extract meaningful choices (e.g. < 2), return empty
    if len(clean_choices) < 2:
        return text, []

    return question_part, clean_choices

def get_label_from_answer(answer_raw):
    """
    Converts 'A', 'B', 'C', 'D' (or 'A. Text') into 0, 1, 2, 3.
    """
    if not answer_raw:
        return -1
    
    # Take just the first character to handle cases like "A. Respirator"
    answer_letter = str(answer_raw).strip()[0].upper()
    
    mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
    return mapping.get(answer_letter, -1)

# --- 1. Load Data ---

# dataset_base = "hf://datasets/yujunzhou/LabSafety_Bench/"
# splits = {'QA_I': 'data/QA_I-00000-of-00001.parquet'}

# NOTE: Replace 'df' loading with your actual load line if different
# df = pd.read_parquet(dataset_base + splits["QA_I"])

# Assuming df is already loaded in your environment based on the error provided:
# df = ... 

# 2. Handle Image Column Renaming
if 'Decoded Image' in df.columns:
    print("Renaming 'Decoded Image' to 'image'...")
    df.rename(columns={'Decoded Image': 'image'}, inplace=True)
elif 'image' not in df.columns:
    print("Warning: Neither 'Decoded Image' nor 'image' column found.")

# Filter: We only want rows that have images
df_with_images = df[df['image'].notna()].copy()

# 3. Generate Metadata
df_with_images['unique_id'] = df_with_images.apply(
    lambda x: generate_hash(f"{x['Question']}_{x['image']}"), axis=1
)

print("Extracting categories...")
df_with_images['first_category'] = df_with_images['Category'].apply(extract_first_category)

# --- 4. Process Questions, Choices, and Labels ---

parsed_data = []
print("Parsing questions, choices, and labels...")

for idx, row in df_with_images.iterrows():
    # Parse Question and Choices
    q_text, choices = parse_question_and_choices(row['Question'])
    
    # Parse Label using 'Correct Answer'
    label = get_label_from_answer(row['Correct Answer'])
    
    # Create new dictionary for the processed row
    row_dict = row.to_dict()
    row_dict['question'] = q_text  # Overwrite with cleaned question text
    row_dict['choices'] = choices
    row_dict['label'] = label
    
    parsed_data.append(row_dict)

# Create DataFrame
df_processed = pd.DataFrame(parsed_data)

# Convert all column names to lowercase
df_processed.columns = df_processed.columns.str.lower()

# --- 5. Split into Question and Knowledge CSVs ---

questions_list = []
knowledges_list = []

# Group by 'first_category' (which is now lowercase in columns -> 'first_category')
grouped = df_processed.groupby('first_category')

print(f"Processing {len(grouped)} categories...")

for category, group in grouped:
    # We need at least 2 items to form a pair
    if len(group) < 2:
        continue

    # 1. Knowledge Source (The LAST item in the category)
    knowledge_source_row = group.iloc[-1]
    
    # 2. Questions (All items EXCEPT the last one)
    problem_rows = group.iloc[:-1]

    for _, row in problem_rows.iterrows():
        # -- Build Question Entry --
        q_data = row.to_dict()
        # Ensure ID is present (it should be 'unique_id' from earlier)
        q_data['id'] = row['unique_id'] 
        questions_list.append(q_data)

        # -- Build Knowledge Entry --
        # Copy the knowledge source
        k_data = knowledge_source_row.to_dict()
        
        # Linkage
        k_data['id'] = knowledge_source_row['unique_id'] # ID of the knowledge item
        k_data['reference_to'] = row['unique_id']        # ID of the question it helps
        k_data['reference_type'] = "similar"
        
        knowledges_list.append(k_data)

# --- 6. Save ---

df_questions = pd.DataFrame(questions_list)
df_knowledges = pd.DataFrame(knowledges_list)

os.makedirs('raw_files', exist_ok=True)
df_questions.to_csv("raw_files/safety_questions.csv", index=False)
df_knowledges.to_csv("raw_files/safety_knowledges.csv", index=False)

print("\n--- Processing Complete ---")
print(f"Total Questions: {len(df_questions)}")
print(f"Total Knowledge Links: {len(df_knowledges)}")
print(f"Columns in output: {list(df_questions.columns)}")

# Verification
if not df_questions.empty:
    print("\nSample Check:")
    print(f"Question: {df_questions.iloc[0]['question'][:50]}...")
    print(f"Choices: {df_questions.iloc[0]['choices']}")
    print(f"Label: {df_questions.iloc[0]['label']}")

Renaming 'Decoded Image' to 'image'...
Extracting categories...
Parsing questions, choices, and labels...
Processing 9 categories...

--- Processing Complete ---
Total Questions: 124
Total Knowledge Links: 124
Columns in output: ['question', 'explanation', 'correct answer', 'category', 'topic', 'image path', 'level', 'image', 'unique_id', 'first_category', 'choices', 'label', 'id']

Sample Check:
Question: Which of the following is necessary to maintain sa...
Choices: ['Using equipment that is regularly calibrated and maintained,', 'Ensuring continuous airflow within the containment area,', 'Wearing minimal protective equipment for convenience,', 'Using HEPA filters for air circulation']
Label: 0
